# OECD FDI Income: MANOVA, MANCOVA, PERMANOVA, and PERMANCOVA

This notebook examines multivariate group effects and effect sizes.

Because the dataset contains one substantive continuous economic outcome (`OBS_VALUE_NUM`) and one native temporal numeric variable (`TIME_PERIOD`), the notebook uses a pragmatic two-response setup for the classical multivariate part:

- `OBS_VALUE_NUM`
- `OBS_VALUE_LOG1P_ABS`, a scale-stabilized companion measure derived from the same numeric outcome

For the distance-based permutation section, all variables are used together through encoded categorical features and standardized numeric features.


In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from statsmodels.multivariate.manova import MANOVA
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from IPython.display import display, Markdown

ALPHA = 0.05
N_PERM = 499


In [2]:
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')
df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)
df = df.dropna(subset=['OBS_VALUE_NUM']).copy()
df['OBS_VALUE_LOG1P_ABS'] = np.log1p(df['OBS_VALUE_NUM'].abs())
display(Markdown(f'**Rows used in multivariate analysis:** {df.shape[0]:,}'))
df[['TYPE_ENTITY', 'MEASURE_PRINCIPLE', 'TIME_PERIOD', 'OBS_VALUE_NUM', 'OBS_VALUE_LOG1P_ABS']].head()


**Rows used in multivariate analysis:** 7,407

,TYPE_ENTITY,MEASURE_PRINCIPLE,TIME_PERIOD,OBS_VALUE_NUM,OBS_VALUE_LOG1P_ABS
1,ALL,DI,2023,1.940070e+13,30.596330
2,ALL,DI,2023,2.001860e+13,30.627683
3,ROU,DO,2024,9.733939e+13,32.209225
10,ALL,DI,2024,2.210346e+13,30.726755
11,ALL,DI,2024,2.982317e+13,31.026307


## Classical MANOVA and MANCOVA

The categorical predictors are `TYPE_ENTITY` and `MEASURE_PRINCIPLE`. The MANCOVA step adds `TIME_PERIOD` as a continuous covariate.


In [3]:
manova_model = MANOVA.from_formula(
    'OBS_VALUE_NUM + OBS_VALUE_LOG1P_ABS ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE)',
    data=df
)
manova_results = manova_model.mv_test()
print(manova_results)


                    Multivariate linear model
                                                                 
-----------------------------------------------------------------
       Intercept        Value  Num DF   Den DF   F Value   Pr > F
-----------------------------------------------------------------
          Wilks' lambda 0.3370 1.0000 7403.0000 14567.3533 0.0000
         Pillai's trace 0.6630 1.0000 7403.0000 14567.3533 0.0000
 Hotelling-Lawley trace 1.9678 1.0000 7403.0000 14567.3533 0.0000
    Roy's greatest root 1.9678 1.0000 7403.0000 14567.3533 0.0000
-----------------------------------------------------------------
                                                                 
-----------------------------------------------------------------
      C(TYPE_ENTITY)     Value  Num DF   Den DF   F Value  Pr > F
-----------------------------------------------------------------
           Wilks' lambda 0.4387 2.0000 7403.0000 4735.3152 0.0000
          Pillai's trace 0.561

In [4]:
mancova_model = MANOVA.from_formula(
    'OBS_VALUE_NUM + OBS_VALUE_LOG1P_ABS ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD',
    data=df
)
mancova_results = mancova_model.mv_test()
print(mancova_results)


                   Multivariate linear model
                                                                
----------------------------------------------------------------
         Intercept        Value  Num DF   Den DF  F Value Pr > F
----------------------------------------------------------------
            Wilks' lambda 0.9992 1.0000 7402.0000  5.6513 0.0175
           Pillai's trace 0.0008 1.0000 7402.0000  5.6513 0.0175
   Hotelling-Lawley trace 0.0008 1.0000 7402.0000  5.6513 0.0175
      Roy's greatest root 0.0008 1.0000 7402.0000  5.6513 0.0175
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
     C(TYPE_ENTITY)     Value  Num DF   Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda 0.4389 2.0000 7402.0000 4730.6740 0.0000
         Pillai's trace 0.5611 2.0000 7402.00

## Follow-up Effect Sizes for Classical Models

Pillai's trace is used as a multivariate effect indicator, and partial eta squared is added from follow-up univariate models for each response.


In [5]:
def partial_eta_squared(anova_table, effect_name):
    ss_effect = anova_table.loc[effect_name, 'sum_sq']
    ss_error = anova_table.loc['Residual', 'sum_sq']
    return ss_effect / (ss_effect + ss_error)


effect_rows = []
for dv in ['OBS_VALUE_NUM', 'OBS_VALUE_LOG1P_ABS']:
    model = ols(f'{dv} ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD', data=df).fit()
    table = anova_lm(model, typ=2)
    for effect in ['C(TYPE_ENTITY)', 'C(MEASURE_PRINCIPLE)', 'TIME_PERIOD']:
        effect_rows.append({
            'dependent_variable': dv,
            'effect': effect,
            'F': table.loc[effect, 'F'],
            'p_value': table.loc[effect, 'PR(>F)'],
            'partial_eta_squared': partial_eta_squared(table, effect),
        })

effect_size_df = pd.DataFrame(effect_rows)
effect_size_df.round(6)


,dependent_variable,effect,F,p_value,partial_eta_squared
0,OBS_VALUE_NUM,C(TYPE_ENTITY),449.535510,0.000000,0.108308
1,OBS_VALUE_NUM,C(MEASURE_PRINCIPLE),0.114208,0.735413,0.000015
2,OBS_VALUE_NUM,TIME_PERIOD,0.689568,0.406338,0.000093
3,OBS_VALUE_LOG1P_ABS,C(TYPE_ENTITY),4637.399507,0.000000,0.556150
4,OBS_VALUE_LOG1P_ABS,C(MEASURE_PRINCIPLE),8.762856,0.003084,0.001182
5,OBS_VALUE_LOG1P_ABS,TIME_PERIOD,5.339998,0.020869,0.000721


## PERMANOVA and PERMANCOVA with All Variables

This section uses all available project variables at once:

- categorical fields are one-hot encoded
- numeric fields are standardized
- the permutation multivariate test is built on an Euclidean-response formulation, which is equivalent to a Euclidean PERMANOVA-style decomposition


In [6]:
categorical_cols = [
    'REF_AREA', 'MEASURE', 'UNIT_MEASURE', 'MEASURE_PRINCIPLE', 'ACCOUNTING_ENTRY',
    'TYPE_ENTITY', 'FDI_COMP', 'SECTOR', 'COUNTERPART_AREA', 'LEVEL_COUNTERPART',
    'ACTIVITY', 'FREQ', 'FDI_COLLECTION_ID', 'OBS_STATUS', 'UNIT_MULT', 'CURRENCY', 'DECIMALS'
]
numeric_cols = ['TIME_PERIOD', 'OBS_VALUE_NUM']

encoded_cat = pd.get_dummies(df[categorical_cols].astype(str), drop_first=False)
scaled_num = df[numeric_cols].copy()
scaled_num = (scaled_num - scaled_num.mean()) / scaled_num.std(ddof=0)
Y = pd.concat([encoded_cat, scaled_num], axis=1).astype(float).to_numpy()
display(Markdown(f'**All-variable response matrix shape:** {Y.shape[0]:,} x {Y.shape[1]:,}'))


**All-variable response matrix shape:** 7,407 x 85

In [7]:
def fit_rss(y, x):
    beta = np.linalg.pinv(x) @ y
    resid = y - x @ beta
    return float(np.sum(resid ** 2))


def design_intercept(data):
    return np.ones((data.shape[0], 1), dtype=float)


def design_factor(data, column):
    return pd.get_dummies(data[column].astype(str), drop_first=True).to_numpy(dtype=float)


def design_covariate(data, column):
    values = data[column].to_numpy(dtype=float)
    values = (values - values.mean()) / values.std(ddof=0)
    return values.reshape(-1, 1)


def freedman_lane_term_test(y, reduced_x, term_x, n_perm=499, seed=42):
    rng = np.random.default_rng(seed)
    full_x = np.column_stack([reduced_x, term_x])
    rss_reduced = fit_rss(y, reduced_x)
    rss_full = fit_rss(y, full_x)
    ss_term = rss_reduced - rss_full
    df_term = np.linalg.matrix_rank(full_x) - np.linalg.matrix_rank(reduced_x)
    df_resid = y.shape[0] - np.linalg.matrix_rank(full_x)
    total_ss = float(np.sum((y - y.mean(axis=0, keepdims=True)) ** 2))
    observed_f = (ss_term / df_term) / (rss_full / df_resid)
    observed_r2 = ss_term / total_ss

    h_reduced = reduced_x @ np.linalg.pinv(reduced_x)
    fitted_reduced = h_reduced @ y
    resid_reduced = y - fitted_reduced

    perm_f = []
    for _ in range(n_perm):
        perm_idx = rng.permutation(y.shape[0])
        y_perm = fitted_reduced + resid_reduced[perm_idx]
        rss_red_p = fit_rss(y_perm, reduced_x)
        rss_full_p = fit_rss(y_perm, full_x)
        ss_term_p = rss_red_p - rss_full_p
        f_p = (ss_term_p / df_term) / (rss_full_p / df_resid)
        perm_f.append(f_p)

    p_value = (np.sum(np.array(perm_f) >= observed_f) + 1) / (n_perm + 1)
    return {
        'df_term': df_term,
        'df_resid': df_resid,
        'pseudo_F': observed_f,
        'R2': observed_r2,
        'p_value': p_value,
    }


In [8]:
intercept = design_intercept(df)
type_x = design_factor(df, 'TYPE_ENTITY')
mp_x = design_factor(df, 'MEASURE_PRINCIPLE')
time_x = design_covariate(df, 'TIME_PERIOD')

permanova_rows = []
reduced_x = intercept
for name, term_x in [('TYPE_ENTITY', type_x), ('MEASURE_PRINCIPLE', mp_x)]:
    result = freedman_lane_term_test(Y, reduced_x, term_x, n_perm=N_PERM, seed=42)
    permanova_rows.append({'model': 'PERMANOVA', 'term': name, **result})
    reduced_x = np.column_stack([reduced_x, term_x])

reduced_x = intercept
for name, term_x in [('TIME_PERIOD', time_x), ('TYPE_ENTITY', type_x), ('MEASURE_PRINCIPLE', mp_x)]:
    result = freedman_lane_term_test(Y, reduced_x, term_x, n_perm=N_PERM, seed=123)
    permanova_rows.append({'model': 'PERMANCOVA', 'term': name, **result})
    reduced_x = np.column_stack([reduced_x, term_x])

perm_df = pd.DataFrame(permanova_rows)
perm_df.round(6)


,model,term,df_term,df_resid,pseudo_F,R2,p_value
0,PERMANOVA,TYPE_ENTITY,2,7404,665.752841,0.152425,0.002
1,PERMANOVA,MEASURE_PRINCIPLE,1,7403,969.243160,0.098123,0.002
2,PERMANCOVA,TIME_PERIOD,1,7405,1813.040293,0.196684,0.002
3,PERMANCOVA,TYPE_ENTITY,2,7403,865.468297,0.152233,0.002
4,PERMANCOVA,MEASURE_PRINCIPLE,1,7402,1313.478152,0.098122,0.002


## Interpretation


In [ ]:
comment = (
    '### Comment\n\n'
    'The classical MANOVA/MANCOVA results provide a model-based multivariate view, while the PERMANOVA/PERMANCOVA '
    'results provide a distance-based permutation view using all variables together. Pillai-style multivariate tests '
    'and R² effect sizes should be read together: statistical significance can appear in large samples even when the '
    'practical effect size is modest. The permutation section is especially useful here because it relaxes strict '
    'distributional assumptions and lets the full mixed-type feature set contribute to the multivariate profile.'
)
display(Markdown(comment))
